In [11]:
import pandas as pd
 
df_raw   = pd.read_csv("../../CSV/compare_region_basin/country_area_timeseries.csv") 
df_area = pd.read_csv("../../CSV/gridarea/Basin_area.csv") 

In [12]:
df_raw = df_raw.rename(columns={
    "basin": "region_name",  # 区域标识
    "area":  "value"         # 数值
})

# ---------- 3. 只保留 region_ / basin_ ----------
df = df_raw[df_raw["variable"].str.startswith(("region_", "basin_"))].copy()

# mode（region / basin）
df["mode"] = df["variable"].str.split("_").str[0]

# 土地类型
df["type_raw"] = df["variable"].str.split("_").str[1]
type_map = {
    "agri": "CL",
    "grassland": "GL",
    "forest": "FRS"
}
df["type"] = df["type_raw"].map(type_map)

# 只保留必要列
df = df[["region_name", "year", "mode", "type", "value"]]

# ---------- 4. pivot 成 wide ----------
df_wide = df.pivot_table(
    index=["region_name", "year", "type"],
    columns="mode",
    values="value"
).reset_index()

# 重命名数值列
df_wide = df_wide.rename(columns={
    "region": "region_value",
    "basin":  "basin_value"
})

# ---------- 5. 计算差值 ----------
df_wide["diff_region-basin"] = (
    df_wide["region_value"] - df_wide["basin_value"]
)
target_years = [2005, 2050, 2100]
df_wide = df_wide[df_wide["year"].isin(target_years)]
# ---------- 6. 处理面积表 ----------
df_area = df_area.rename(columns={"Value": "area"})

# 统一区域标识（和 df_wide 对齐）
df_area["region_name"] = df_area["country"] + "_" + df_area["basin"]

df_area = df_area[["region_name", "area"]]

# ---------- 7. merge ----------
df_out = df_wide.merge(df_area, on="region_name", how="left")

df_out["diff/area"] = df_out["diff_region-basin"] / df_out["area"]

In [13]:
num_cols = ["region_value", "basin_value", "diff_region-basin", "diff/area"]
df_out[num_cols] = df_out[num_cols].applymap(
    lambda x: f"{x:.3f}" if pd.notnull(x) else x
)

df_out.to_csv(
    "../../CSV/compare_region_basin/region_basin_diff(nc-nc).csv",
    index=False
)

print("Saved: region_basin_diff.csv")


Saved: region_basin_diff.csv


C:\Users\ZSR\AppData\Local\Temp\ipykernel_18024\2406353190.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_out[num_cols] = df_out[num_cols].applymap(
